# שיעור 08 - תבנית עיצוב רב-סוכנית


## התקנה


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## למה מערכות רב-סוכנים?

משימות מהחיים האמיתיים כמו תכנון טיולים כוללות הרבה סוגי מומחיות שונים — לוגיסטיקה, ידע מקומי, תקצוב ועוד. סוכן יחיד שמנסה לטפל בכל דבר במהירות הופך לבלתי ניתן לניהול.

מערכות רב-סוכנים פותרות זאת דרך **התמחות**: כל סוכן מתמקד בתחום מומחיות אחד, ומייצר תוצאות באיכות גבוהה יותר מאשר גנרליסט. הן גם משפרות את ה**סקאלביליות** — אפשר להוסיף סוכנים חדשים (למשל, מומחה לטיסות, מבקר מסעדות) בלי לשכתב את תהליך העבודה הקיים. הסוכנים מתרכבים יחד דרך צינור מובנה, ומעבירים הקשר מאחד לשני.


## יצירת סוכנים מיוחדים


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## בניית זרימת עבודה רציפה

`WorkflowBuilder` מאפשר לך לחבר סוכנים לגרף מכוון. כאן אנו יוצרים צינור דו-שלבי פשוט: ה-**TravelPlanner** מנסח את המסלול, ואז ה-**TravelConcierge** מבקר ומשפר אותו.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## הוספת סוכנים נוספים לזרימת העבודה

אחד היתרונות הגדולים ביותר של דפוס הסוכנים המרובים הוא כמה קל להרחיב אותו. למטה נוסיף סוכן **BudgetReviewer** שבודק את התוכנית מול התקציב של הנוסע, מסמן פריטים שעשויים לדחוף את העלויות מעל הגבול, ומציע חלופות לחיסכון בכסף. זרימת העבודה כעת מריצה שלושה סוכנים ברצף:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## סיכום

בשיעור זה למדת כיצד:

1. **ליצור סוכנים מתמחים** — כל אחד עם תפקיד ממוקד (תכנון, קונסיירז', סקירת תקציב).
2. **לחבר סוכנים לזרימת עבודה סדרתית** באמצעות `WorkflowBuilder` ו-`add_edge`.
3. **לשדר פלט** מצינור רב-סוכני, ולעקוב מי הסוכן המדבר.
4. **להרחיב זרימת עבודה** על ידי הוספת סוכנים חדשים לשרשרת מבלי לשנות את הקיימים.

דפוס העיצוב רב-הסוכני שומר על כל סוכן פשוט תוך הפקת תוצאות עשירות ומבוקרות יותר היטב מאשר כל סוכן יחיד יכול להשיג לבדו.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:  
מסמך זה תורגם באמצעות שירות התרגום בינה מלאכותית [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לדעת כי תרגומים ממוחשבים עלולים להכיל שגיאות או אי-דיוקים. המסמך המקורי בשפת המקור שלו צריך להיחשב כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי אנושי. איננו אחראים לכל אי-הבנה או פרשנות שגויה הנובעת מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
